# 🐍 Notebook 01 · Python interview basics

Practice for **Part 1** of your visual guide (cards 1.1 → 1.16, plus a bonus). These are the warm-up questions a "straightforward" Python exam usually opens with.

**How to use:** run **▶ Run me first**, then for each exercise write your code → **Shift + Enter** → run the `check(...)` cell.
Try to finish each one in **under 5 minutes**, talking out loud as you go.

> 💾 **Opened this from GitHub?** Colab won't keep your edits until you choose **File → Save a copy in Drive**. Do that first.

In [ ]:
#@title ▶ Run me first: loads the practice data and the checker { display-mode: "form" }
import math, random, statistics, inspect, functools, csv, io, os, warnings
from collections import Counter, defaultdict, deque
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
_G = globals()
TESTS, DONE = {}, set()

def need(name):
    if name not in _G:
        raise NameError(f"I can't find `{name}`. Run your code cell first, and use exactly that name.")
    return _G[name]

def value(name):
    v = need(name)
    if v is Ellipsis:
        raise NotImplementedError
    return v

def close(a, b, tol=1e-6):
    if a is None:
        raise AssertionError("a result is None. Did you forget `return`?")
    return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))

def expect(fn_name, args, expected, tol=None, **kw):
    f = need(fn_name)
    got = f(*args, **kw)
    shown = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()])
    if tol is not None:
        ok = got is not None and close(got, expected, tol)
    else:
        ok = got == expected
    hint = "  (Did you forget `return`?)" if got is None and expected is not None else ""
    assert ok, f"{fn_name}({shown}) should return {expected!r}, but returned {got!r}.{hint}"
    return got

def check(qid):
    try:
        TESTS[qid]()
    except NotImplementedError:
        print(f"✏️  Exercise {qid}: replace the ... with your code, run that cell, then run this check again.")
    except AssertionError as e:
        print(f"❌ Exercise {qid}: not yet. {e}")
    except NameError as e:
        print(f"⚠️  Exercise {qid}: {e}")
    except Exception as e:
        print(f"💥 Exercise {qid}: your code raised {type(e).__name__}: {e}\n   Read the message, fix your cell, re-run it, then check again. (Seeing this before you've written anything is normal.)")
    else:
        DONE.add(qid)
        print(f"✅ Exercise {qid}: all checks passed!")

def progress():
    def key(k):
        return [float(p) if p.replace('.', '', 1).isdigit() else 999 for p in k.lstrip('M').split('.')]
    keys = sorted(TESTS, key=key)
    print(f"{len(DONE)}/{len(TESTS)} exercises passed")
    print("  ".join(("✅ " if k in DONE else "⬜ ") + k for k in keys))

# ---------- practice data ----------
# no data needed for this notebook

# ---------- checks (no peeking needed: solutions are below each exercise) ----------
def _test_1_1():
    f = need("add_reading")
    a, b = f(1), f(2)
    assert b == [2], f"second call add_reading(2) returned {b}; it should be [2] (no leftovers from call 1)"
    assert f(3, [1]) == [1, 3], "passing your own list should still work: add_reading(3, [1]) → [1, 3]"
    default = inspect.signature(f).parameters["readings"].default
    assert default is None, f"the default should be None, not {default!r}"
TESTS['1.1'] = _test_1_1

def _test_1_2():
    expect("has_duplicates", (["M1", "M2", "M1"],), True)
    expect("has_duplicates", (["M1", "M2"],), False)
    expect("has_duplicates", ([],), False)
    expect("common_assets", (["T3", "T1", "T2"], ["T2", "T9", "T3"]), ["T2", "T3"])
    big = list(range(200000))
    import time; t0 = time.perf_counter(); need("has_duplicates")(big); dt = time.perf_counter() - t0
    assert dt < 1.0, f"has_duplicates took {dt:.1f}s on 200,000 IDs. Use a set instead of nested loops."
TESTS['1.2'] = _test_1_2

def _test_1_3():
    codes = ["E12", "E07", "E12", "E31", "E07", "E12"]
    expect("top_codes", (codes, 2), [("E12", 3), ("E07", 2)])
    expect("top_codes", (codes, 1), [("E12", 3)])
TESTS['1.3'] = _test_1_3

def _test_1_4():
    rows = [("T1", 71.2), ("T2", 65.0), ("T1", 74.8), ("T2", 66.1), ("T1", 73.0)]
    expect("mean_by_asset", (rows,), {"T1": 73.0, "T2": 65.55})
    expect("mean_by_asset", ([],), {})
TESTS['1.4'] = _test_1_4

def _test_1_5():
    lines = ["S1,2026-01-01,1.5", "S1,2026-01-02,ERR", "S2,2026-01-01,2.0", "broken line", "S3,2026-01-03,-0.5\n"]
    g = need("read_valid")(lines)
    assert inspect.isgenerator(g), "read_valid should be a generator: use `yield`, not `return` a list"
    got = list(g)
    assert got == [("S1", 1.5), ("S2", 2.0), ("S3", -0.5)], f"got {got}"
TESTS['1.5'] = _test_1_5

def _test_1_6():
    def ref(values, w, th):
        return [i for i in range(w - 1, len(values)) if sum(values[i - w + 1:i + 1]) / w > th]
    for args in [([1, 2, 3, 10, 11, 12, 2, 1], 3, 8), ([1, 2, 3, 10, 11, 12, 2, 1], 2, 10), ([5, 5, 5], 4, 1), ([9], 1, 8)]:
        expect("rolling_alarms", args, ref(*args))
TESTS['1.6'] = _test_1_6

def _test_1_7():
    expect("merge_outages", ([(9, 12), (1, 4), (15, 16), (2, 6), (8, 10)],), [[1, 6], [8, 12], [15, 16]])
    expect("merge_outages", ([(1, 10), (2, 3)],), [[1, 10]])
    expect("merge_outages", ([(1, 2), (2, 3)],), [[1, 3]])
    expect("merge_outages", ([],), [])
    expect("total_outage", ([(9, 12), (1, 4), (15, 16), (2, 6), (8, 10)],), 10)
TESTS['1.7'] = _test_1_7

def _test_1_8():
    assert value("a_after") == [1, 2, [3, 4, 6], 5], "a_after: b IS a (so the 5 lands in a), and the inner list [3, 4] is shared with c (so the 6 lands there too)"
    assert value("c_after") == [1, 2, [3, 4, 6]], "c_after: .copy() made a new outer list (no 5), but the inner list is shared (it has the 6)"
    assert value("a_is_b") is True and value("a_is_c") is False, "b = a gives the same object; .copy() gives a different one"
    assert value("a_eq_c") is False, "a has an extra 5 at the end, so the contents differ"
TESTS['1.8'] = _test_1_8

def _test_1_9():
    deco = need("count_calls")
    @deco
    def add(a, b=1):
        return a + b
    assert add.calls == 0, "calls should start at 0"
    assert add(2) == 3 and add(2, b=5) == 7 and add(0) == 1, "the decorated function should still return its normal result"
    assert add.calls == 3, f"after 3 calls, .calls should be 3, got {add.calls}"
    assert add.__name__ == "add", "use @functools.wraps(func) so the name is kept"
TESTS['1.9'] = _test_1_9

def _test_1_10():
    expect("to_fahrenheit", ([61.2, None, 70.5, 88.1, None, 64.0],), [142.2, 158.9, 190.6, 147.2])
    expect("to_fahrenheit", ([0, None],), [32.0])
    expect("old_assets_dict", ({"T1": 31, "T2": 12, "T3": 40}, 30), {"T1": 31, "T3": 40})
TESTS['1.10'] = _test_1_10

def _test_1_11():
    assets = [("T1", 0.82, 31), ("T2", 0.40, 12), ("T3", 0.82, 40), ("T4", 0.15, 5), ("T5", 0.61, 22)]
    expect("rank_assets", (assets,), ["T3", "T1", "T5", "T2", "T4"])
TESTS['1.11'] = _test_1_11

def _test_1_12():
    expect("parse_line", ("2026-01-10 14:00:00|T2|TEMP_HIGH|95.0",), {"ts": "2026-01-10 14:00:00", "asset": "T2", "code": "TEMP_HIGH", "value": 95.0})
    expect("parse_line", (" 2026-01-11 09:00:00 | T7 | LOAD_LOW | 12 \n",), {"ts": "2026-01-11 09:00:00", "asset": "T7", "code": "LOAD_LOW", "value": 12.0})
TESTS['1.12'] = _test_1_12

def _test_1_13():
    expect("safe_ratio", (1, 3), 0.333)
    expect("safe_ratio", (1, 0), None)
    try:
        need("safe_ratio")("a", 2)
    except ValueError:
        pass
    else:
        raise AssertionError('safe_ratio("a", 2) should raise ValueError')
TESTS['1.13'] = _test_1_13

def _test_1_14():
    A = need("Asset")
    t1, t2 = A("T1", 31), A("T2", 12)
    assert t1.asset_id == "T1" and t1.age_years == 31, "store asset_id and age_years on self"
    assert t1.mean_reading() is None, "mean_reading() with no readings should return None"
    t1.add_reading(60); t1.add_reading(70)
    assert t1.mean_reading() == 65, f"mean_reading() should be 65, got {t1.mean_reading()}"
    assert t2.readings == [], "each object needs its own readings list (create it inside __init__)"
    assert t1.is_old() is True and t2.is_old() is False and t2.is_old(threshold=10) is True, "is_old() looks wrong"
TESTS['1.14'] = _test_1_14

def _test_1_15():
    expect("pair_with_total", ([120, 340, 75, 260, 410], 600), (1, 3))
    expect("pair_with_total", ([300, 5], 600), None)
    expect("pair_with_total", ([300, 5, 300], 600), (0, 2))
    expect("pair_with_total", ([], 1), None)
TESTS['1.15'] = _test_1_15

def _test_1_16():
    expect("second_largest", ([5, 9, 9, 3, 7],), 7)
    expect("second_largest", ([4, 4],), None)
    expect("second_largest", ([],), None)
    expect("dedupe", (["T2", "T1", "T2", "T3", "T1"],), ["T2", "T1", "T3"])
TESTS['1.16'] = _test_1_16

def _test_1_17():
    expect("word_counts", ("Grid down! Grid up, grid DOWN.",), {"grid": 3, "down": 2, "up": 1})
    expect("word_counts", ("",), {})
TESTS['1.17'] = _test_1_17

print("✅ Setup complete: 17 exercises ready. Work top to bottom.")

### Exercise 1.1 · Fix the mutable default bug
<small>📖 Card 1.1 in the visual guide</small>

This function has the classic bug: calling it twice leaks old readings into the new call. Fix it so every call without a list starts fresh.

In [ ]:
def add_reading(x, readings=[]):      # 🐞
    readings.append(x)
    return readings

In [ ]:
check("1.1")

<details><summary>💡 Show a solution</summary>

```python
def add_reading(x, readings=None):
    if readings is None:
        readings = []
    readings.append(x)
    return readings
```

</details>

### Exercise 1.2 · Sets for fast membership
<small>📖 Card 1.2 in the visual guide</small>

**a)** `has_duplicates(ids)` returns True if any meter ID appears more than once.

**b)** `common_assets(a, b)` returns a **sorted list** of IDs that appear in both lists.

Both should be O(n): use sets, not nested loops.

In [ ]:
def has_duplicates(ids):
    ...

def common_assets(a, b):
    ...

In [ ]:
check("1.2")

<details><summary>💡 Show a solution</summary>

```python
def has_duplicates(ids):
    return len(set(ids)) != len(ids)

def common_assets(a, b):
    return sorted(set(a) & set(b))
```

</details>

### 🗣 Say it out loud · 1.2s

**“Why is checking `x in my_set` faster than `x in my_list`?”**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

A list is checked item by item, so the time grows with its length: O(n). A set stores items in a hash table: Python computes the item's hash and jumps straight to where it would be, so on average it's O(1) regardless of size. If I'm checking membership inside a loop, converting to a set first turns O(n²) into O(n).

</details>

### Exercise 1.3 · Top fault codes with Counter
<small>📖 Card 1.3 in the visual guide</small>

Write `top_codes(codes, k)` that returns the `k` most frequent codes as a list of `(code, count)` tuples, most frequent first.

In [ ]:
from collections import Counter

def top_codes(codes, k):
    ...

In [ ]:
check("1.3")

<details><summary>💡 Show a solution</summary>

```python
from collections import Counter

def top_codes(codes, k):
    return Counter(codes).most_common(k)
```

</details>

### Exercise 1.4 · Group and average without pandas
<small>📖 Card 1.4 in the visual guide</small>

Write `mean_by_asset(rows)` where `rows` is a list of `(asset, temp)` tuples. Return a dict `{asset: mean temp}`, rounded to 2 decimals.

<details><summary>🔎 Hint</summary>

`defaultdict(list)`, then a dict comprehension: `{a: round(sum(v) / len(v), 2) for a, v in acc.items()}`.

</details>

In [ ]:
from collections import defaultdict

def mean_by_asset(rows):
    ...

In [ ]:
check("1.4")

<details><summary>💡 Show a solution</summary>

```python
from collections import defaultdict

def mean_by_asset(rows):
    acc = defaultdict(list)
    for asset, temp in rows:
        acc[asset].append(temp)
    return {a: round(sum(v) / len(v), 2) for a, v in acc.items()}
```

</details>

### Exercise 1.5 · A generator that skips bad lines
<small>📖 Card 1.5 in the visual guide</small>

Write a **generator** `read_valid(lines)`. Each line looks like `"S1,2026-01-01,1.5"`.
For each line, `yield` a tuple `(sensor, value_as_float)`. Skip lines whose value isn't a number, and lines that don't have exactly 3 parts.

In [ ]:
def read_valid(lines):
    ...

In [ ]:
check("1.5")

<details><summary>💡 Show a solution</summary>

```python
def read_valid(lines):
    for line in lines:
        parts = line.strip().split(",")
        if len(parts) != 3:
            continue
        sensor, ts, value = parts
        try:
            yield sensor, float(value)
        except ValueError:
            continue
```

</details>

### Exercise 1.6 · Rolling-average alarm
<small>📖 Card 1.6 in the visual guide</small>

Write `rolling_alarms(values, window, threshold)` that returns the list of **indexes** where the average of the last `window`
values (ending at that index) is greater than `threshold`. Only check indexes where a full window exists.
Bonus: keep a running total so it's O(n).

In [ ]:
from collections import deque

def rolling_alarms(values, window, threshold):
    ...

In [ ]:
check("1.6")

<details><summary>💡 Show a solution</summary>

```python
from collections import deque

def rolling_alarms(values, window, threshold):
    buf, total, alarms = deque(), 0.0, []
    for i, v in enumerate(values):
        buf.append(v)
        total += v
        if len(buf) > window:
            total -= buf.popleft()
        if len(buf) == window and total / window > threshold:
            alarms.append(i)
    return alarms
```

</details>

### Exercise 1.7 · Merge outage intervals
<small>📖 Card 1.7 in the visual guide</small>

**a)** `merge_outages(intervals)` merges overlapping `(start, end)` intervals and returns a list of `[start, end]` lists, sorted by start.
Intervals that touch (end == start) count as overlapping.

**b)** `total_outage(intervals)` returns the total hours covered (after merging).

In [ ]:
def merge_outages(intervals):
    ...

def total_outage(intervals):
    ...

In [ ]:
check("1.7")

<details><summary>💡 Show a solution</summary>

```python
def merge_outages(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

def total_outage(intervals):
    return sum(end - start for start, end in merge_outages(intervals))
```

</details>

### Exercise 1.8 · Predict the output: references and copies
<small>📖 Card 1.8 in the visual guide</small>

**Don't run this code.** Read it and predict the results:

```python
a = [1, 2, [3, 4]]
b = a
c = a.copy()
b.append(5)
c[2].append(6)
```

Fill in your predictions: `a_after` (the value of `a` at the end), `c_after`, `a_is_b` (`a is b`), `a_is_c` (`a is c`), `a_eq_c` (`a == c`).

In [ ]:
a_after = ...
c_after = ...
a_is_b = ...
a_is_c = ...
a_eq_c = ...

In [ ]:
check("1.8")

<details><summary>💡 Show a solution</summary>

```python
a_after = [1, 2, [3, 4, 6], 5]   # b is the same object as a; the inner list is shared with c
c_after = [1, 2, [3, 4, 6]]      # c's outer list is separate, so it didn't get the 5
a_is_b = True
a_is_c = False
a_eq_c = False
```

</details>

### Exercise 1.9 · Write a decorator
<small>📖 Card 1.9 in the visual guide</small>

Write a decorator `count_calls` that counts how many times the decorated function has been called. Store the count on the wrapper
as an attribute `.calls` (starting at 0). The function must still return its normal result and keep its name (use `functools.wraps`).

<details><summary>🔎 Hint</summary>

Inside `count_calls(func)`, define `wrapper(*args, **kwargs)` that does `wrapper.calls += 1` and returns `func(*args, **kwargs)`. Set `wrapper.calls = 0` before returning wrapper.

</details>

In [ ]:
import functools

def count_calls(func):
    ...

In [ ]:
check("1.9")

<details><summary>💡 Show a solution</summary>

```python
import functools

def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper
```

</details>

### Exercise 1.10 · Comprehensions
<small>📖 Card 1.10 in the visual guide</small>

Write each as a **one-line** comprehension:

**a)** `to_fahrenheit(readings)` returns a list of Fahrenheit values rounded to 1 decimal, **skipping `None`**.

**b)** `old_assets_dict(ages, min_age)` returns a dict with only the assets older than `min_age`.

In [ ]:
def to_fahrenheit(readings):
    ...

def old_assets_dict(ages, min_age):
    ...

In [ ]:
check("1.10")

<details><summary>💡 Show a solution</summary>

```python
def to_fahrenheit(readings):
    return [round(t * 9 / 5 + 32, 1) for t in readings if t is not None]

def old_assets_dict(ages, min_age):
    return {a: age for a, age in ages.items() if age > min_age}
```

</details>

### Exercise 1.11 · Sort with a key
<small>📖 Card 1.11 in the visual guide</small>

`assets` is a list of `(id, risk, age)` tuples. Write `rank_assets(assets)` that returns the **list of IDs** ordered by
risk (highest first); ties are broken by age (oldest first).

In [ ]:
def rank_assets(assets):
    ...

In [ ]:
check("1.11")

<details><summary>💡 Show a solution</summary>

```python
def rank_assets(assets):
    return [a[0] for a in sorted(assets, key=lambda a: (-a[1], -a[2]))]
```

</details>

### Exercise 1.12 · Parse a log line
<small>📖 Card 1.12 in the visual guide</small>

Write `parse_line(line)` for lines like `"2026-01-10 14:00:00|T2|TEMP_HIGH|95.0"`. Return a dict with keys
`ts`, `asset`, `code` and `value` (value as a float). Ignore spaces around the pieces.

In [ ]:
def parse_line(line):
    ...

In [ ]:
check("1.12")

<details><summary>💡 Show a solution</summary>

```python
def parse_line(line):
    ts, asset, code, value = [p.strip() for p in line.split("|")]
    return {"ts": ts, "asset": asset, "code": code, "value": float(value)}
```

</details>

### Exercise 1.13 · try / except / else
<small>📖 Card 1.13 in the visual guide</small>

Write `safe_ratio(a, b)`:
- returns `a / b` rounded to 3 decimals,
- returns `None` if `b` is zero,
- raises a `ValueError` (with any message) if the inputs aren't numbers.

In [ ]:
def safe_ratio(a, b):
    ...

In [ ]:
check("1.13")

<details><summary>💡 Show a solution</summary>

```python
def safe_ratio(a, b):
    try:
        r = a / b
    except ZeroDivisionError:
        return None
    except TypeError as e:
        raise ValueError(f"bad input: {e}")
    else:
        return round(r, 3)
```

</details>

### Exercise 1.14 · A small class
<small>📖 Card 1.14 in the visual guide</small>

Write a class `Asset` with:
- `__init__(self, asset_id, age_years)`, storing both, plus an empty list `self.readings`
- `add_reading(self, value)`, which appends to readings
- `mean_reading(self)`, which returns the average, or `None` if there are no readings
- `is_old(self, threshold=30)`, which returns True if age_years is above threshold

In [ ]:
class Asset:
    ...

In [ ]:
check("1.14")

<details><summary>💡 Show a solution</summary>

```python
class Asset:
    def __init__(self, asset_id, age_years):
        self.asset_id = asset_id
        self.age_years = age_years
        self.readings = []

    def add_reading(self, value):
        self.readings.append(value)

    def mean_reading(self):
        return sum(self.readings) / len(self.readings) if self.readings else None

    def is_old(self, threshold=30):
        return self.age_years > threshold
```

</details>

### Exercise 1.15 · Two-sum in one pass
<small>📖 Card 1.15 in the visual guide</small>

Write `pair_with_total(loads, target)` that returns a tuple `(i, j)` with `i < j` such that `loads[i] + loads[j] == target`,
or `None` if there's no such pair. Aim for O(n) with a dict.

In [ ]:
def pair_with_total(loads, target):
    ...

In [ ]:
check("1.15")

<details><summary>💡 Show a solution</summary>

```python
def pair_with_total(loads, target):
    seen = {}
    for i, x in enumerate(loads):
        need_ = target - x
        if need_ in seen:
            return seen[need_], i
        seen[x] = i
    return None
```

</details>

### 🗣 Say it out loud · 1.15s

**“Walk me through the time and space complexity of your two-sum solution. What's the brute-force alternative?”**

*Answer out loud first, or type your answer in the cell below. Then compare.*

✍️ *Your answer (double-click to edit):*



<details><summary>💬 Show a model answer</summary>

Brute force checks every pair with two nested loops, so the time is O(n²) and the extra memory is O(1). My solution walks the list once, and for each number checks whether its partner (target − x) is already in a dict of numbers seen so far. Dict lookups are O(1) on average, so the total time is O(n), at the cost of O(n) extra memory for the dict. That's the classic trade of memory for time.

</details>

### Exercise 1.16 · Second largest and order-preserving dedupe
<small>📖 Card 1.16 in the visual guide</small>

**a)** `second_largest(values)` returns the second-largest **distinct** value, or `None` if there are fewer than 2 distinct values.

**b)** `dedupe(items)` removes duplicates but keeps the **original order** of first appearances.

In [ ]:
def second_largest(values):
    ...

def dedupe(items):
    ...

In [ ]:
check("1.16")

<details><summary>💡 Show a solution</summary>

```python
def second_largest(values):
    distinct = sorted(set(values))
    return distinct[-2] if len(distinct) >= 2 else None

def dedupe(items):
    return list(dict.fromkeys(items))
```

</details>

### Exercise 1.17 · Bonus: word counts
<small>📖 Card 1.12 in the visual guide</small>

Write `word_counts(text)` that returns a dict of how often each word appears, **case-insensitive**, ignoring the punctuation `. , ! ?`

<details><summary>🔎 Hint</summary>

Lower-case the text, replace each punctuation mark with a space (or nothing), then `.split()` and count.

</details>

In [ ]:
def word_counts(text):
    ...

In [ ]:
check("1.17")

<details><summary>💡 Show a solution</summary>

```python
from collections import Counter

def word_counts(text):
    for p in ".,!?":
        text = text.replace(p, " ")
    return dict(Counter(text.lower().split()))
```

</details>

## 🏁 Your progress
Run the cell below to see which exercises you've passed.

In [ ]:
progress()